# Quantum Fourier Transform

The Quantum Fourier Transform (QFT) is the quantum analog of the discrete Fourier transform. It maps the computational basis states to the Fourier basis:

$$\text{QFT}|j\rangle = \frac{1}{\sqrt{N}} \sum_{k=0}^{N-1} e^{2\pi i jk/N} |k\rangle$$

The QFT circuit uses $O(n^2)$ gates (Hadamard + controlled rotations), compared to $O(n \cdot 2^n)$ for the classical FFT.

In [ ]:
import cudaq
import numpy as np


@cudaq.kernel
def qft_3q(input_val: int):
    """3-qubit QFT. input_val encodes the input state via X gates."""
    qubits = cudaq.qvector(3)
    if (input_val & 1) != 0:
        x(qubits[0])
    if (input_val & 2) != 0:
        x(qubits[1])
    if (input_val & 4) != 0:
        x(qubits[2])
    h(qubits[0])
    crz(qubits[1], qubits[0], np.pi / 2)
    crz(qubits[2], qubits[0], np.pi / 4)
    h(qubits[1])
    crz(qubits[2], qubits[1], np.pi / 2)
    h(qubits[2])
    swap(qubits[0], qubits[2])

In [ ]:
basis = ["|000>", "|001>", "|010>", "|011>",
         "|100>", "|101>", "|110>", "|111>"]

for val in range(4):
    sv = np.array(cudaq.get_state(qft_3q, val))
    probs = np.abs(sv) ** 2
    print(f"QFT|{val:03d}>:")
    for b, amp, prob in zip(basis, sv, probs):
        if prob > 0.001:
            print(f"  {b}: amp={amp:+.4f}  P={prob:.4f}")
    print()

Each input state $|j\rangle$ produces a uniform superposition with phases that encode $e^{2\pi i jk/8}$. The QFT is the key subroutine inside Quantum Phase Estimation (QPE).